In [3]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
import joblib
import warnings

# ปิดแจ้งเตือนจุกจิก
warnings.filterwarnings("ignore")

print("⚡ กำลังเริ่มกระบวนการรวมร่างข้อมูล (Data Merging)...")

# --- 1. โหลดข้อมูลเก่า ---
print("📥 1. โหลดข้อมูลเก่า (zeus_train.csv)...")
try:
    df_old = pd.read_csv('zeus_train.csv')
    # FIX 1: แก้ปัญหาอ่านเวลาไม่ออก (format='mixed', dayfirst=True)
    df_old['datetime'] = pd.to_datetime(df_old['datetime'], format='mixed', dayfirst=True, errors='coerce')
    df_old = df_old.dropna(subset=['datetime']) # ทิ้งแถวที่เวลาพังจริงๆ
    
    df_old['hour'] = df_old['datetime'].dt.hour
    df_old['is_day'] = df_old['hour'].apply(lambda x: 1 if 6 <= x <= 18 else 0)
    df_old['rain_clean'] = df_old['rain'].apply(lambda x: 1 if x > 0.05 else 0)
    print(f"   ✅ ได้ข้อมูลเก่ามา {len(df_old)} แถว")
except Exception as e:
    print(f"   ❌ โหลดข้อมูลเก่าไม่สำเร็จ: {e}")
    df_old = pd.DataFrame()

# --- 2. โหลดข้อมูลใหม่ ---
print("📥 2. โหลดข้อมูลใหม่ (zeus_dataset_combined_new.csv)...")
try:
    df_new = pd.read_csv('zeus_dataset_combined_new.csv')
    df_new['datetime'] = pd.to_datetime(df_new['datetime'], format='mixed', dayfirst=True, errors='coerce')
    df_new = df_new.dropna(subset=['datetime'])
    
    df_new['hour'] = df_new['datetime'].dt.hour
    df_new['is_day'] = df_new['hour'].apply(lambda x: 1 if 6 <= x <= 18 else 0)
    df_new['rain_clean'] = df_new['rain'].apply(lambda x: 1 if x > 0.05 else 0)
    print(f"   ✅ ได้ข้อมูลใหม่มา {len(df_new)} แถว")
except Exception as e:
    print(f"   ❌ โหลดข้อมูลใหม่ไม่สำเร็จ: {e}")
    df_new = pd.DataFrame()

# --- 3. รวมร่างข้อมูล (Concatenate) ---
features_to_keep = ['temp', 'humidity', 'pressure', 'rain', 'uv', 'wind_speed', 'hour', 'is_day', 'rain_clean']

# คัดเฉพาะ Dataframe ที่มีข้อมูล (ป้องกันปัญหาเอา Dataframe ว่างมารวม)
dfs_to_concat = []
if not df_old.empty: dfs_to_concat.append(df_old[features_to_keep])
if not df_new.empty: dfs_to_concat.append(df_new[features_to_keep])

if len(dfs_to_concat) > 0:
    df_combined = pd.concat(dfs_to_concat, ignore_index=True)
    
    # FIX 2: บังคับ Data Type ไม่ให้เพี้ยน ป้องกัน Classifier พัง
    df_combined = df_combined.dropna() # ทิ้งค่าว่างที่อาจหลงเหลือจากการรวมร่าง
    df_combined['rain_clean'] = df_combined['rain_clean'].astype(int)
    df_combined['is_day'] = df_combined['is_day'].astype(int)
    
    print(f"\n✨ รวมร่างสำเร็จ! ตอนนี้ Zeus มีข้อมูลให้เรียนรู้ทั้งหมด {len(df_combined)} แถว!")
else:
    print("\n❌ ไม่มีข้อมูลให้เทรนเลย กรุณาตรวจสอบไฟล์ CSV!")
    exit()

# --- 4. เริ่มฝึกสมอง AI ใหม่ด้วยข้อมูลที่ใหญ่ขึ้น ---
print("\n🧠 4. กำลังส่งเทพ Zeus เข้าโรงเรียน (Training AI Models)...")

# Model 1: The Oracle (อุณหภูมิ)
print("   ⏳ กำลังฝึก: โมเดลอุณหภูมิ (Temperature)...")
X_temp = df_combined[['humidity', 'pressure', 'rain', 'uv', 'wind_speed', 'hour', 'is_day']]
y_temp = df_combined['temp']
model_temp = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model_temp.fit(X_temp, y_temp)
joblib.dump(model_temp, 'zeus_oracle_model.pkl')

# Model 2: ความชื้น (Humidity)
print("   ⏳ กำลังฝึก: โมเดลความชื้น (Humidity)...")
X_hum = df_combined[['temp', 'pressure', 'rain', 'uv', 'wind_speed', 'hour', 'is_day']]
y_hum = df_combined['humidity']
model_hum = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model_hum.fit(X_hum, y_hum)
joblib.dump(model_hum, 'zeus_humidity_model.pkl')

# Model 3: โอกาสเกิดฝน (Rain Probability)
print("   ⏳ กำลังฝึก: โมเดลทำนายฝน (Rain Classifier)...")
X_rain = df_combined[['temp', 'humidity', 'pressure', 'uv', 'wind_speed', 'hour', 'is_day']]
y_rain = df_combined['rain_clean']
model_rain = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
model_rain.fit(X_rain, y_rain)
joblib.dump(model_rain, 'zeus_rain_class_model.pkl')

# Model 4: รังสี UV
print("   ⏳ กำลังฝึก: โมเดลรังสี UV...")
X_uv = df_combined[['temp', 'humidity', 'pressure', 'rain', 'wind_speed', 'hour', 'is_day']]
y_uv = df_combined['uv']
model_uv = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model_uv.fit(X_uv, y_uv)
joblib.dump(model_uv, 'zeus_uv_model.pkl')

print("\n🎉 อัปเกรดสมองเสร็จสมบูรณ์")

⚡ กำลังเริ่มกระบวนการรวมร่างข้อมูล (Data Merging)...
📥 1. โหลดข้อมูลเก่า (zeus_train.csv)...
   ✅ ได้ข้อมูลเก่ามา 39207 แถว
📥 2. โหลดข้อมูลใหม่ (zeus_dataset_combined_new.csv)...
   ✅ ได้ข้อมูลใหม่มา 9475 แถว

✨ รวมร่างสำเร็จ! ตอนนี้ Zeus มีข้อมูลให้เรียนรู้ทั้งหมด 48682 แถว!

🧠 4. กำลังส่งเทพ Zeus เข้าโรงเรียน (Training AI Models)...
   ⏳ กำลังฝึก: โมเดลอุณหภูมิ (Temperature)...
   ⏳ กำลังฝึก: โมเดลความชื้น (Humidity)...
   ⏳ กำลังฝึก: โมเดลทำนายฝน (Rain Classifier)...
   ⏳ กำลังฝึก: โมเดลรังสี UV...

🎉 อัปเกรดสมองเสร็จสมบูรณ์
